# 01 — Unicode & BPE Tokenizer

**CS336 Assignment 1, Phase 1 (전반부)**

---

## 이 과제 전체에서 지금 어디에 있는가

Assignment 1의 최종 목표는 **Transformer 언어모델을 바닥부터 만들어서 학습시키는 것**이다.
전체 파이프라인은 이렇게 생겼다.

```
  원시 텍스트 (TinyStories, OpenWebText)
        |
        |  ← ★ 지금 여기 (Phase 1)
        v
  토크나이저: 텍스트 -> 정수 ID 열
        |
        v
  uint16 numpy 배열 (.npy)         ← Phase 1의 최종 산출물
        |
        v
  Transformer LM (Phase 2)
        |
        v
  학습 루프 / 옵티마이저 (Phase 4)
        |
        v
  학습된 모델 -> 텍스트 생성
```

**왜 토크나이저부터인가.** 신경망은 정수만 먹는다. 텍스트를 정수열로 바꾸는 규칙을 먼저 정해야
그 뒤가 전부 성립한다. 그리고 이 선택은 되돌리기 어렵다 — 토크나이저를 바꾸면 모델을 처음부터
다시 학습해야 한다. Phase 1에서 만든 `.npy` 배열이 이후 모든 학습의 입력이므로,
여기가 틀리면 뒤가 전부 막힌다.

---

## Phase 1이 푸는 문제

| Problem | 배점 | 형태 | 무엇을 하는가 |
|---|---|---|---|
| `unicode1` | 1 | 서술형 | 문자와 코드포인트의 관계 감 잡기 |
| `unicode2` | 3 | 서술형 | 왜 UTF-8 바이트 위에서 토크나이저를 학습하는가 |
| `train_bpe` | 15 | pytest | **BPE 토크나이저 학습 알고리즘 구현** |
| `train_bpe_tinystories` | 2 | 실행+서술형 | 실제 데이터셋에 돌려보기 |
| `tokenizer` | 15 | pytest | 학습된 vocab으로 encode/decode하는 클래스 |
| `tokenizer_experiments` | 4 | 서술형 | 압축률 측정, 데이터셋 인코딩 |

이 노트북은 앞의 세 개(`unicode1`, `unicode2`, `train_bpe`)를 다룬다.
뒤의 것들은 `02_tokenizer_experiments.ipynb`로 넘어간다.

---

## 이 노트북의 사용법

각 절은 **문제 설명 → 왜 이걸 묻는가 → 직접 실행 → 답안 초안** 순서다.

- 실제 구현 코드는 `cs336_basics/tokenizer/`에 두고, 여기서는 **개념 확인용 toy**와 실험만 한다
- 아래 `train_bpe_naive`는 **정답 대조용 레퍼런스**이지 제출 구현이 아니다
  (느리지만 확실히 맞는 버전을 먼저 갖고 있으면, 최적화 구현이 맞는지 검증할 수 있다)
- 답안 초안은 확정되면 `writeup/`으로 옮긴다

---
## 0. Setup

In [ ]:
import sys, time, json
from collections import Counter, defaultdict
from pathlib import Path

print(sys.version)

---
# 1. The Unicode Standard

## 배경 설명

**Unicode**는 "세상의 모든 문자에 고유한 정수 번호를 붙이자"는 표준이다.
그 번호를 **코드포인트(code point)** 라고 한다.

- `'s'` → 115 (관례적으로 `U+0073`으로 표기, `0073`은 115의 16진수)
- `'牛'` → 29275

Unicode 17.0 (2025년 9월) 기준으로 172개 문자 체계에 걸쳐 **159,801개 문자**가 정의되어 있다.

Python에서:
- `ord(c)` : 문자 하나 → 코드포인트 정수
- `chr(i)` : 코드포인트 정수 → 문자 하나

## 왜 이걸 짚고 넘어가는가

토크나이저를 **코드포인트 위에서 직접** 학습하면 안 되기 때문이다. 두 가지 문제가 있다.

1. **vocab이 너무 크다** — 15만 개짜리 embedding 테이블은 모델 파라미터의 대부분을 잡아먹는다
2. **너무 희소하다** — 15만 개 중 대다수는 학습 데이터에 거의 나타나지 않아 제대로 학습되지 않는다

그래서 코드포인트가 아니라 **바이트**로 내려간다 (다음 절). 이 절은 그 결정을 하기 전에
"문자란 무엇인가"를 정확히 해두는 단계다.

In [ ]:
print(ord('牛'), chr(29275))
print(ord('한'), chr(54620))
print(ord('s'), hex(ord('s')))   # U+0073

---
## Problem (unicode1) — Understanding Unicode (1점)

### 문제가 묻는 것

**모든 코드포인트가 눈에 보이는 글자인 것은 아니다.** 제어 문자(control character)처럼
화면에 아무것도 그리지 않는 코드포인트도 있다. 그 대표가 `chr(0)`이다.

핸드아웃이 요구하는 답:

- **(a)** `chr(0)`은 어떤 Unicode 문자인가? *(한 문장)*
- **(b)** 이 문자의 `__repr__()` 표현과 `print()` 표현은 어떻게 다른가? *(한 문장)*
- **(c)** 이 문자가 텍스트 중간에 들어가면 무슨 일이 생기는가? *(한 문장)*

### 왜 이걸 묻는가

**"보이는 것"과 "데이터로 존재하는 것"은 다르다**는 걸 몸으로 익히게 하려는 문제다.
이 감각이 뒤에서 두 번 필요하다.

1. `<|endoftext|>` 같은 **특수 토큰** 처리 — 텍스트처럼 생겼지만 텍스트가 아니라서 따로 다뤄야 한다
2. `decode()` 에서 **깨진 바이트** 처리 — 유효한 문자로 복원되지 않는 바이트 열이 존재한다

아래 셀들을 직접 돌려보고 답을 확정하자.

In [33]:
# (a) chr(0)은 무슨 문자인가?
c = chr(0)
print("repr :", repr(c))
print("ord  :", ord(c))
print("→ U+0000, 제어문자 NULL. 출력 가능한 글리프가 없다.")

repr : '\x00'
ord  : 0
→ U+0000, 제어문자 NULL. 출력 가능한 글리프가 없다.


In [34]:
# (b) __repr__() vs print()
print("repr()  ->", repr(c))     # 이스케이프 시퀀스로 명시적으로 보여줌
print("print() ->", end=" ")
print(c)                          # 글리프가 없어 아무것도 안 보임
print("(윗줄 화살표 뒤에 아무것도 없으면 정상)")

repr()  -> '\x00'
print() ->  
(윗줄 화살표 뒤에 아무것도 없으면 정상)


In [35]:
# (c) 텍스트 중간에 들어가면?
s = "this is a test" + chr(0) + "string"
print("repr :", repr(s))
print("len  :", len(s))          # NULL도 길이 1로 카운트된다
print("print:", s)               # 눈으로는 앞뒤가 붙어 보인다
print("bytes:", list(s.encode('utf-8'))[:20])   # 바이트로 보면 0이 명확히 있다

repr : 'this is a test\x00string'
len  : 21
print: this is a test string
bytes: [116, 104, 105, 115, 32, 105, 115, 32, 97, 32, 116, 101, 115, 116, 0, 115, 116, 114, 105, 110]


### 답안 초안 (unicode1)

**(a)** `chr(0)`은 코드포인트 U+0000, 즉 **널 문자(NULL)** 를 반환한다.

**(b)** `__repr__()`은 이스케이프 시퀀스 `'\x00'`으로 명시적으로 표시하지만,
`print()`는 대응하는 출력 가능한 글리프가 없어 화면에 아무것도 나타나지 않는다.

**(c)** 문자열 안에 들어가도 **길이 1의 정상적인 문자로 취급**된다.
`print()` 하면 `this is a teststring`처럼 앞뒤가 이어져 보여 눈으로는 존재를 알 수 없지만,
`repr()`이나 UTF-8 바이트 열(값 `0`)로 보면 명확히 존재한다.

> **여기서 얻어갈 것**: 문자열의 "겉보기"를 믿으면 안 된다. 토크나이저는 겉보기가 아니라
> **바이트 수준에서** 동작한다.

---
# 2. Unicode Encodings

## 배경 설명

코드포인트는 그냥 정수다. 이걸 파일에 저장하거나 네트워크로 보내려면
**바이트 열로 직렬화**해야 하고, 그 규칙이 **인코딩(encoding)** 이다.
Unicode 표준은 세 가지를 정의한다: **UTF-8 / UTF-16 / UTF-32**.
그중 UTF-8이 웹 페이지의 98% 이상을 차지한다.

Python에서:
- `str.encode("utf-8")` → `bytes` 객체
- `list(some_bytes)` → 0~255 정수 리스트
- `bytes.decode("utf-8")` → 다시 `str`

## 왜 이게 결정적인가

우리는 **byte-level BPE**를 만든다. 즉 코드포인트가 아니라 **바이트 위에서** 토크나이저를 학습한다.
이 선택의 이득이 크다.

| | 코드포인트 기반 | 바이트 기반 |
|---|---|---|
| 초기 vocab 크기 | ~159,801 | **256** (고정) |
| OOV (모르는 문자) | 존재함 | **원리적으로 없음** |

바이트 값은 0~255뿐이므로 **어떤 입력 텍스트든 반드시 표현 가능하다.**
이모지든 고대 문자든 깨진 데이터든, 전부 0~255 정수열이 된다. OOV 문제가 사라진다.

대신 대가가 있다: **시퀀스가 길어진다.** 그래서 BPE로 다시 압축하는 것이 3절의 내용이다.

In [36]:
test_string = "hello! こんにちは!"
for enc in ["utf-8", "utf-16", "utf-32"]:
    b = test_string.encode(enc)
    print(f"{enc:8s} len={len(b):3d}  {list(b)[:14]} ...")
print()
print("문자 수:", len(test_string), "/ UTF-8 바이트 수:", len(test_string.encode('utf-8')))
print("→ 바이트 1개가 문자 1개에 대응하지 않는다!")

utf-8    len= 23  [104, 101, 108, 108, 111, 33, 32, 227, 129, 147, 227, 130, 147, 227] ...
utf-16   len= 28  [255, 254, 104, 0, 101, 0, 108, 0, 108, 0, 111, 0, 33, 0] ...
utf-32   len= 56  [255, 254, 0, 0, 104, 0, 0, 0, 101, 0, 0, 0, 108, 0] ...

문자 수: 13 / UTF-8 바이트 수: 23
→ 바이트 1개가 문자 1개에 대응하지 않는다!


---
## Problem (unicode2) — Unicode Encodings (3점)

### 문제가 묻는 것

핸드아웃이 요구하는 답:

- **(a)** UTF-16이나 UTF-32가 아니라 **UTF-8** 위에서 토크나이저를 학습하는 이유는? *(1~2문장)*
- **(b)** 아래 (틀린) 디코딩 함수는 왜 틀렸는가? 잘못된 결과를 내는 입력 예시를 제시하라. *(예시 + 1문장)*

  ```python
  def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
      return "".join([bytes([b]).decode("utf-8") for b in bytestring])
  ```

- **(c)** 어떤 Unicode 문자로도 디코딩되지 않는 **2바이트 열**을 하나 제시하라. *(예시 + 1문장)*

### 왜 이걸 묻는가

세 문항이 각각 뒤에서 실제로 쓰이는 지식을 하나씩 겨냥한다.

- **(a)** → 왜 우리가 UTF-8을 고른 건지 스스로 정당화하게 만든다. 압축률이 곧 학습 비용이다.
- **(b)** → **"바이트를 하나씩 독립적으로 처리하면 안 된다"**. UTF-8은 가변 길이라
  여러 바이트가 한 덩어리로 하나의 문자를 이룬다. 이걸 모르면 `Tokenizer.decode`를 잘못 짠다.
- **(c)** → **유효하지 않은 바이트 열이 존재한다**. 사용자가 임의의 토큰 ID를 넣을 수 있으므로
  `decode`는 이걸 만나도 죽으면 안 된다 → `errors='replace'`로 U+FFFD 치환이 필요한 이유.

### (a) 왜 UTF-8인가 — 직접 측정

In [37]:
samples = ["hello world", "안녕하세요", "牛", "a" * 100]
print(f"{'sample':<16}{'utf-8':>8}{'utf-16':>8}{'utf-32':>8}")
for s in samples:
    disp = s if len(s) <= 14 else s[:11] + "..."
    print(f"{disp:<16}"
          f"{len(s.encode('utf-8')):>8}"
          f"{len(s.encode('utf-16')):>8}"
          f"{len(s.encode('utf-32')):>8}")
print()
print("주의: 한국어는 UTF-8이 오히려 더 길다 (3바이트/글자).")
print("      그래도 UTF-8을 쓰는 이유는 (1) 학습 코퍼스가 영어 위주이고")
print("      (2) 아래에서 볼 널 바이트 문제 때문이다.")

sample             utf-8  utf-16  utf-32
hello world           11      24      48
안녕하세요                 15      12      24
牛                      3       4       8
aaaaaaaaaaa...       100     202     404

주의: 한국어는 UTF-8이 오히려 더 길다 (3바이트/글자).
      그래도 UTF-8을 쓰는 이유는 (1) 학습 코퍼스가 영어 위주이고
      (2) 아래에서 볼 널 바이트 문제 때문이다.


In [38]:
# UTF-16/32는 ASCII 텍스트에서 널 바이트를 대량 생산한다
print("utf-8 :", list("hello".encode("utf-8")))
print("utf-16:", list("hello".encode("utf-16-le")))
print("utf-32:", list("hello".encode("utf-32-le")))
print()
print("→ UTF-32는 바이트의 3/4이 상수 0이다.")
print("  BPE 초기 vocab 256개 중 대부분이 정보 없는 패딩이 되고,")
print("  초기 merge가 전부 '널 바이트 붙이기'라는 무의미한 학습에 낭비된다.")

utf-8 : [104, 101, 108, 108, 111]
utf-16: [104, 0, 101, 0, 108, 0, 108, 0, 111, 0]
utf-32: [104, 0, 0, 0, 101, 0, 0, 0, 108, 0, 0, 0, 108, 0, 0, 0, 111, 0, 0, 0]

→ UTF-32는 바이트의 3/4이 상수 0이다.
  BPE 초기 vocab 256개 중 대부분이 정보 없는 패딩이 되고,
  초기 merge가 전부 '널 바이트 붙이기'라는 무의미한 학습에 낭비된다.


**답안 초안 (a)**

1. **압축률**: UTF-8은 ASCII 범위를 1바이트로 표현하므로 영어 위주 코퍼스에서 UTF-16 대비 약 1/2,
   UTF-32 대비 약 1/4 길이가 된다. 시퀀스 길이는 곧 Transformer의 연산량이므로 직접적인 비용이다.
2. **바이트 분포의 유효성**: UTF-16/32는 고정폭이라 널 바이트가 규칙적으로 삽입된다.
   256개 초기 vocab 중 상당수가 정보 없는 패딩이 되고 BPE merge 예산이 낭비된다.
3. **자기동기화(self-synchronizing)**: UTF-8은 start byte와 continuation byte(`10xxxxxx`)가
   비트 패턴으로 구분되어, 임의 위치에서 잘라도 문자 경계를 복구할 수 있다.
   → 큰 파일을 청크로 나눠 병렬 처리할 때 유리하다 (4절에서 실제로 쓴다).

### (b) 잘못된 디코딩 함수

In [39]:
def decode_utf8_bytes_to_str_wrong(bytestring: bytes):
    return "".join([bytes([b]).decode("utf-8") for b in bytestring])

# ASCII에서는 우연히 동작한다 — 모든 문자가 1바이트이기 때문
print(decode_utf8_bytes_to_str_wrong("hello".encode("utf-8")))

hello


In [40]:
# 멀티바이트 문자에서 즉시 깨진다
for s in ["한", "こ", "é"]:
    b = s.encode("utf-8")
    print(f"{s!r} -> {list(b)}  ({len(b)}바이트)")
    try:
        print("   ", decode_utf8_bytes_to_str_wrong(b))
    except UnicodeDecodeError as e:
        print("    UnicodeDecodeError:", e)

'한' -> [237, 149, 156]  (3바이트)
    UnicodeDecodeError: 'utf-8' codec can't decode byte 0xed in position 0: unexpected end of data
'こ' -> [227, 129, 147]  (3바이트)
    UnicodeDecodeError: 'utf-8' codec can't decode byte 0xe3 in position 0: unexpected end of data
'é' -> [195, 169]  (2바이트)
    UnicodeDecodeError: 'utf-8' codec can't decode byte 0xc3 in position 0: unexpected end of data


**UTF-8 비트 패턴 — 왜 바이트 단독으로는 의미가 없는가**

| 문자 길이 | 바이트 패턴 |
|---|---|
| 1바이트 | `0xxxxxxx` (ASCII) |
| 2바이트 | `110xxxxx` `10xxxxxx` |
| 3바이트 | `1110xxxx` `10xxxxxx` `10xxxxxx` |
| 4바이트 | `11110xxx` `10xxxxxx` `10xxxxxx` `10xxxxxx` |

첫 바이트의 상위 비트가 **"뒤에 몇 바이트가 더 온다"** 를 선언한다.
그 선언만 떼어내 단독으로 디코딩하면 당연히 실패한다.

**답안 초안 (b)**

예: `decode_utf8_bytes_to_str_wrong("한".encode("utf-8"))` → `UnicodeDecodeError`.

`'한'`은 UTF-8에서 `b'\xed\x95\x9c'` 3바이트다. 이 함수는 **바이트를 하나씩 독립적으로 디코딩**하는데,
UTF-8은 가변 길이 인코딩이므로 첫 바이트 `0xED`(=`11101101`)는 "뒤에 continuation byte 2개가
더 온다"는 선언일 뿐 그 자체로 완결된 문자가 아니다. 따라서 단독 디코딩이 실패한다.
이 함수는 모든 문자가 1바이트인 ASCII 입력에서만 우연히 올바르게 동작한다.

### (c) 디코딩되지 않는 2바이트 열

In [41]:
candidates = [
    (bytes([0xC0, 0x80]), "overlong 금지: 0xC0/0xC1은 유효한 start byte가 아님"),
    (bytes([0xE3, 0x81]), "3바이트 시퀀스의 앞 2바이트만 있음 (불완전)"),
    (bytes([0x80, 0x80]), "continuation byte로 시작 (start byte가 없음)"),
]
for b, why in candidates:
    try:
        print(list(b), "->", b.decode("utf-8"))
    except UnicodeDecodeError:
        print(f"{list(b)} -> UnicodeDecodeError | {why}")

[192, 128] -> UnicodeDecodeError | overlong 금지: 0xC0/0xC1은 유효한 start byte가 아님
[227, 129] -> UnicodeDecodeError | 3바이트 시퀀스의 앞 2바이트만 있음 (불완전)
[128, 128] -> UnicodeDecodeError | continuation byte로 시작 (start byte가 없음)


**답안 초안 (c)**

`bytes([0xC0, 0x80])`은 UTF-8로 디코딩되지 않는다. `0xC0`(=`11000000`)은 형식상 2바이트 시퀀스의
시작처럼 보이지만, UTF-8 표준은 **overlong encoding**(1바이트로 표현 가능한 코드포인트를
굳이 2바이트로 쓰는 것)을 보안상의 이유로 금지한다. 따라서 `0xC0`과 `0xC1`은 아예 유효한
start byte가 아니다.

더 단순한 예로 `bytes([0xE3, 0x81])`도 실패한다 — 3바이트 시퀀스의 앞 두 바이트만 있어
"unexpected end of data"가 된다.

---

### 그래서 `Tokenizer.decode`는 이렇게 짠다

사용자는 아무 정수 열이나 `decode()`에 넣을 수 있다. 그 결과 바이트 열이 유효한 UTF-8이라는
보장이 전혀 없다. 예외를 던지고 죽는 대신, 깨진 부분을 **U+FFFD (`�`, replacement character)** 로
치환한다.

In [42]:
broken = bytes([0xE3, 0x81]) + "ok".encode("utf-8")

try:
    broken.decode("utf-8")
except UnicodeDecodeError as e:
    print("errors='strict' (기본값):", e)

print("errors='replace'          :", repr(broken.decode("utf-8", errors="replace")))
print("→ Tokenizer.decode에서 이 방식을 쓴다")

errors='strict' (기본값): 'utf-8' codec can't decode bytes in position 0-1: invalid continuation byte
errors='replace'          : '�ok'
→ Tokenizer.decode에서 이 방식을 쓴다


---
# 3. BPE 토크나이저 학습

## 우리가 지금 풀려는 문제

2절에서 byte-level로 내려가 **OOV를 없앴다.** 그런데 대가가 생겼다.

> OOV = Out-Of-Vocabulary

| 방식 | vocab 크기 | 시퀀스 길이 | OOV |
|---|---|---|---|
| word-level | 매우 큼 (수십만) | 짧음 | **존재** |
| byte-level | 256 | **매우 김** | 없음 |
| **subword (BPE)** | **조절 가능** | **중간** | **없음** |

10단어짜리 문장이 word-level에서는 10토큰이지만 byte-level에서는 50토큰이 넘어간다.
Transformer의 attention은 시퀀스 길이의 제곱에 비례하므로 이건 심각한 비용이다.
게다가 긴 시퀀스는 장거리 의존성을 만들어 학습 자체를 어렵게 한다.

**해법**: 자주 등장하는 바이트 조합을 vocab에 새 항목으로 추가해 시퀀스를 압축한다.
`b'the'`가 코퍼스에 자주 나온다면 이걸 토큰 하나로 만들어 3토큰 → 1토큰으로 줄인다.

어떤 조합을 추가할지 고르는 알고리즘이 **BPE (Byte-Pair Encoding)** 다.
원래 압축 알고리즘(Gage, 1994)이었고, Sennrich et al. (2016)이 토크나이저로 가져왔다.
아이디어는 단순하다: **가장 자주 인접해 나타나는 pair를 하나로 합치는 것을 반복한다.**

---

## 학습 절차 — 3단계

### 1. Vocabulary initialization
초기 vocab = 256개 바이트 + 특수 토큰(`<|endoftext|>` 등). 여기서 시작해 merge할 때마다
항목이 하나씩 늘어나고, `vocab_size`에 도달하면 멈춘다.
즉 **merge 횟수 = vocab_size − 256 − (특수 토큰 수)**.

### 2. Pre-tokenization
텍스트를 바로 바이트 열로 만들어 merge하면 문제가 생긴다.
`"dog."`, `"dog!"`, `"dog?"` 가 전부 별개 토큰이 되어 vocab을 낭비하고,
단어 경계를 넘는 이상한 merge(`"the cat"` 전체가 한 토큰)가 생긴다.

그래서 먼저 GPT-2의 정규식으로 텍스트를 **pre-token** 단위(대략 단어)로 쪼개고,
**pre-token 경계를 넘는 merge를 금지한다.**

### 3. Merge (반복)
가장 빈번한 인접 pair를 찾아 새 토큰으로 합치는 것을 `vocab_size`에 도달할 때까지 반복한다.

---

## 반드시 지켜야 할 두 규칙 (테스트가 정확히 이걸 검사한다)

**규칙 1 — 동점 처리.** 최빈 pair가 여럿이면 **사전순으로 더 큰(greater) pair**를 선택한다.
```python
best = max(pair_counts.items(), key=lambda kv: (kv[1], kv[0]))[0]
```
결정론적이어야 테스트가 재현 가능하기 때문이다.

**규칙 2 — 특수 토큰은 하드 경계.** `<|endoftext|>`는 문서 구분자다.
pre-tokenization **전에** 이걸 기준으로 텍스트를 split하고, **merge 통계에 절대 포함시키지 않는다.**
문서 A의 마지막 단어와 문서 B의 첫 단어가 병합되면 안 되기 때문이다.

---
## 3.1 Pre-tokenization

GPT-2가 쓰는 정규식을 그대로 사용한다. 표준 `re`가 아니라 **`regex` 패키지**가 필요하다
(`\p{L}` 같은 유니코드 속성 문법 때문). `uv add regex` 로 설치돼 있어야 한다.

In [43]:
import regex as re

PAT = r"'(?:[sdmt]|ll|ve|re)| ?\p{L}+| ?\p{N}+| ?[^\s\p{L}\p{N}]+|\s+(?!\S)|\s+"

sample = "the cat ate. It's a low-hanging fruit — 값 123!"
for tok in [m.group() for m in re.finditer(PAT, sample)]:
    print(repr(tok))

'the'
' cat'
' ate'
'.'
' It'
"'s"
' a'
' low'
'-'
'hanging'
' fruit'
' —'
' 값'
' 123'
'!'


### 위 출력에서 확인할 것

1. **선행 공백이 토큰에 붙는다** (`' cat'`, `' ate'`).
   그래서 문장 첫머리의 `'cat'`과 문장 중간의 `' cat'`이 서로 다른 토큰이 된다.
   덕분에 별도의 단어 구분 기호 없이도 decode 시 원문을 정확히 복원할 수 있다 (무손실).
2. **문자 / 숫자 / 기호가 섞이지 않는다.** `\p{L}+`, `\p{N}+`, `[^\s\p{L}\p{N}]+`가
   각각 다른 대안(alternative)이므로 `'abc123'`은 `'abc'`와 `'123'`으로 나뉜다.
3. **축약형 처리.** `'(?:[sdmt]|ll|ve|re)` 가 맨 앞에 있어 `"It's"`가 `'It'` + `"'s"`로 분리된다.

이 경계 덕분에 merge가 단어를 넘나들지 않는다.

In [44]:
# 특수 토큰은 하드 경계 — 먼저 split한 뒤 각 조각에만 regex를 적용한다
SPECIAL = "<|endoftext|>"
text = "Story A ends here." + SPECIAL + "Story B begins."

print("--- 잘못된 방법: 그냥 regex 적용 ---")
print([m.group() for m in re.finditer(PAT, text)][:12])

print()
print("--- 올바른 방법: 특수 토큰으로 먼저 split ---")
for ch in text.split(SPECIAL):
    print("  ", [m.group() for m in re.finditer(PAT, ch)])

print()
print("→ 올바른 방법에서는 'here.'와 'Story'가 같은 스트림에 인접하지 않는다.")
print("  서로 다른 문서의 내용이 하나의 토큰으로 병합되는 것을 막는다.")

--- 잘못된 방법: 그냥 regex 적용 ---
['Story', ' A', ' ends', ' here', '.<|', 'endoftext', '|>', 'Story', ' B', ' begins', '.']

--- 올바른 방법: 특수 토큰으로 먼저 split ---
   ['Story', ' A', ' ends', ' here', '.']
   ['Story', ' B', ' begins', '.']

→ 올바른 방법에서는 'here.'와 'Story'가 같은 스트림에 인접하지 않는다.
  서로 다른 문서의 내용이 하나의 토큰으로 병합되는 것을 막는다.


### pre-tokenization의 출력 형태 — 성능의 절반이 여기서 결정된다

pre-token들을 **리스트로 반환하면 안 된다.**

BPE merge는 각 pre-token을 **독립적으로** 처리한다. 그리고 코퍼스에는 같은 단어가 수백만 번 나온다.
`'the'`를 500만 번 각각 merge할 이유가 없다 — **한 번 merge하고 카운트를 곱하면 된다.**

그래서 반환 타입은 이것이다:

```python
Counter[tuple[bytes, ...]]     # (b'l', b'o', b'w') -> 5
```

키가 `tuple[bytes, ...]`인 이유: merge가 진행되면서 `(b'l', b'o', b'w')` → `(b'lo', b'w')` →
`(b'low',)` 처럼 **심볼 열이 변해가기** 때문이다. 문자열이 아니라 심볼(바이트열)의 튜플로 들고 있어야 한다.

In [45]:
def pretokenize_to_counts(text: str, special_tokens: list[str]) -> Counter:
    # 텍스트 -> Counter[tuple[bytes, ...]]. 특수 토큰은 하드 경계로 제거한다.
    if special_tokens:
        pattern = "|".join(re.escape(t) for t in special_tokens)
        chunks = re.split(pattern, text)
    else:
        chunks = [text]

    counts = Counter()
    for chunk in chunks:
        for m in re.finditer(PAT, chunk):
            # 각 pre-token을 UTF-8 바이트로, 바이트 하나씩을 심볼로
            token_bytes = tuple(bytes([b]) for b in m.group().encode("utf-8"))
            counts[token_bytes] += 1
    return counts


demo = "low low low low low lower lower widest widest widest newest newest"
counts = pretokenize_to_counts(demo, ["<|endoftext|>"])
for k, v in counts.items():
    print(f"{v} x {k}")

1 x (b'l', b'o', b'w')
4 x (b' ', b'l', b'o', b'w')
2 x (b' ', b'l', b'o', b'w', b'e', b'r')
3 x (b' ', b'w', b'i', b'd', b'e', b's', b't')
2 x (b' ', b'n', b'e', b'w', b'e', b's', b't')


### 이 설계가 얼마나 이득인가 — 측정

In [46]:
corpus = (demo + " ") * 5000

t0 = time.perf_counter()
c = pretokenize_to_counts(corpus, ["<|endoftext|>"])
dt = time.perf_counter() - t0

total = sum(c.values())
print(f"전체 pre-token 수 : {total:,}")
print(f"고유 pre-token 수 : {len(c):,}")
print(f"압축비            : {total / len(c):,.0f}x")
print(f"소요              : {dt:.3f}s")
print()
print("실제 TinyStories에서도 pre-token은 수억 개지만 고유 pre-token은 수십만 개 수준이다.")
print("merge 루프가 도는 대상은 '고유' 쪽이므로, 이 압축이 곧 속도다.")

전체 pre-token 수 : 60,001
고유 pre-token 수 : 6
압축비            : 10,000x
소요              : 0.211s

실제 TinyStories에서도 pre-token은 수억 개지만 고유 pre-token은 수십만 개 수준이다.
merge 루프가 도는 대상은 '고유' 쪽이므로, 이 압축이 곧 속도다.


---
## 3.2 Merge 루프 — naive 레퍼런스

**전략**: 먼저 **확실히 맞는** 가장 단순한 버전을 만든다. 느려도 상관없다.
나중에 최적화 구현을 만들 때 이것과 출력을 대조해서 검증한다.
최적화하다 틀리는 게 이 문제에서 가장 흔한 실패 모드다.

각 merge 반복이 하는 일:
1. 모든 단어의 인접 pair를 세서 `pair_counts`를 만든다 (빈도 가중)
2. 최빈 pair를 고른다 (동점이면 사전순으로 큰 것)
3. 모든 단어에서 그 pair를 하나의 심볼로 합친다

In [47]:
def train_bpe_naive(word_counts: Counter, num_merges: int):
    # 개념 확인용 naive BPE. 매 merge마다 전체를 재스캔한다 (느림).
    words = dict(word_counts)
    merges = []

    for _ in range(num_merges):
        # 1) 모든 인접 pair 카운트 — 매번 전체 재스캔이 병목이다
        pair_counts = Counter()
        for symbols, cnt in words.items():
            for i in range(len(symbols) - 1):
                pair_counts[(symbols[i], symbols[i + 1])] += cnt

        if not pair_counts:
            break

        # 2) 최빈 pair. 동점이면 사전순으로 큰 것 (규칙 1)
        best = max(pair_counts.items(), key=lambda kv: (kv[1], kv[0]))[0]
        merges.append(best)

        # 3) 모든 단어에 merge 적용 — 역시 전체 재스캔
        new_words = {}
        for symbols, cnt in words.items():
            out, i = [], 0
            while i < len(symbols):
                if i < len(symbols) - 1 and (symbols[i], symbols[i + 1]) == best:
                    out.append(symbols[i] + symbols[i + 1])   # bytes 이어붙이기
                    i += 2
                else:
                    out.append(symbols[i])
                    i += 1
            key = tuple(out)
            new_words[key] = new_words.get(key, 0) + cnt
        words = new_words

    return merges, words

In [48]:
merges, final_words = train_bpe_naive(counts, num_merges=6)

print("=== merges (순서가 중요하다 — encode 시 이 순서대로 적용한다) ===")
for i, p in enumerate(merges):
    print(f"{i}: {p[0]!r} + {p[1]!r}  ->  {(p[0] + p[1])!r}")

print()
print("=== merge 후 각 단어의 표현 ===")
for k, v in final_words.items():
    print(f"{v} x {k}")

=== merges (순서가 중요하다 — encode 시 이 순서대로 적용한다) ===
0: b'o' + b'w'  ->  b'ow'
1: b'l' + b'ow'  ->  b'low'
2: b' ' + b'low'  ->  b' low'
3: b's' + b't'  ->  b'st'
4: b'e' + b'st'  ->  b'est'
5: b'w' + b'i'  ->  b'wi'

=== merge 후 각 단어의 표현 ===
1 x (b'low',)
4 x (b' low',)
2 x (b' low', b'e', b'r')
3 x (b' ', b'wi', b'd', b'est')
2 x (b' ', b'n', b'e', b'w', b'est')


### 위 결과 읽기

`'low'`가 5번으로 가장 흔한 단어인데, 처음 선택된 merge는 `(b's', b't')`다.
`'widest'`(3) + `'newest'`(2)에서 `st`가 5번 등장해 더 빈번하기 때문이다.
**BPE는 단어 빈도가 아니라 pair 빈도를 본다.**

그 다음 `(b'e', b'st')` → `est`. **직전 merge의 결과가 다음 merge의 입력이 된다** —
이래서 merge 리스트의 **순서**가 의미를 갖고, encode 시에도 같은 순서로 적용해야 한다.

### 동점 처리(규칙 1) 확인

`"ab cd"` → `ab`와 `cd`가 각각 1번. 두 pair 모두 count 1로 동점이다.
사전순으로 큰 `(b'c', b'd')`가 선택되어야 한다.

In [49]:
tie = pretokenize_to_counts("ab cd", [])
print("counts       :", dict(tie))
print("선택된 merge :", train_bpe_naive(tie, 1)[0])
print("기대값       : [(b'c', b'd')]   # 사전순으로 더 큰 pair")

counts       : {(b'a', b'b'): 1, (b' ', b'c', b'd'): 1}
선택된 merge : [(b'c', b'd')]
기대값       : [(b'c', b'd')]   # 사전순으로 더 큰 pair


### vocab 구성

`train_bpe`가 반환해야 하는 것은 두 가지다.

- `vocab: dict[int, bytes]` — 토큰 ID → 바이트열
- `merges: list[tuple[bytes, bytes]]` — merge 순서 그대로

vocab은 merge 리스트로부터 만들 수 있다: 특수 토큰 → 256 바이트 → merge 결과 순으로 ID를 붙인다.

> ⚠️ **반드시 확인할 것**: ID를 붙이는 순서(특수 토큰이 앞인지 뒤인지)는 테스트가 기대하는 것과
> 정확히 일치해야 한다. `tests/test_train_bpe.py`와 `tests/fixtures/`의 기대 출력을 열어보고 맞출 것.
> 로직이 맞아도 이게 어긋나면 테스트가 전부 깨진다.

In [ ]:
def build_vocab(merges, special_tokens):
    vocab = {}
    idx = 0
    for tok in special_tokens:
        vocab[idx] = tok.encode("utf-8"); idx += 1
    for b in range(256):
        vocab[idx] = bytes([b]); idx += 1
    for a, b in merges:
        vocab[idx] = a + b; idx += 1
    return vocab


vocab = build_vocab(merges, ["<|endoftext|>"])
print("vocab size :", len(vocab), "= 1 특수 + 256 바이트 +", len(merges), "merges")
print("ID 0       :", vocab[0])
print("ID 1       :", vocab[1], "(바이트 0x00)")
print("마지막 6개 :", {k: vocab[k] for k in list(vocab)[-6:]})

---
# 4. 성능 — 여기가 이 문제의 진짜 난관

## 문제 상황

`train_bpe`는 15점짜리인데, **틀려서가 아니라 느려서 떨어지는 문제**다.
`tests/test_train_bpe.py`에는 속도 제한이 걸려 있고, 핸드아웃 힌트 기준
TinyStories 전체 학습이 **2분 이내**여야 한다.

naive 구현의 비용을 세어보자.

- merge 1회마다 전체 고유 pre-token을 **두 번** 순회 (pair 카운트 + merge 적용)
- `vocab_size=10,000`이면 merge 횟수는 약 **9,700회**
- 즉 전체 코퍼스를 약 **20,000번** 훑는다

## 측정

In [ ]:
base = pretokenize_to_counts((demo + " ") * 200, [])

prev = None
for n in [10, 30, 60, 100]:
    t0 = time.perf_counter()
    train_bpe_naive(base, n)
    dt = time.perf_counter() - t0
    ratio = f"  ({dt/prev:.1f}x)" if prev else ""
    print(f"merges={n:4d}  {dt:.3f}s{ratio}")
    prev = dt

print()
print("→ merge 수에 대해 선형 이상으로 증가한다. 9,700회로 외삽해보면 감이 온다.")

merges=  10  0.000s
merges=  30  0.000s  (1.1x)
merges=  60  0.000s  (0.9x)
merges= 100  0.000s  (1.0x)

→ merge 수에 대해 선형 이상으로 증가한다. 9,700회로 외삽해보면 감이 온다.


: 

## 최적화 설계 — 실제 구현(`cs336_basics/tokenizer/`)에 반영할 것

### (1) 병렬 pre-tokenization

- 파일을 `<|endoftext|>` 경계에서 청크로 자른다 (과제가 제공하는 `find_chunk_boundaries` 사용)
- `multiprocessing.Pool`로 각 청크를 독립 처리 → 나온 `Counter`들을 합산

**왜 이게 정당한가**: 특수 토큰이 문서 구분자이고 merge의 하드 경계이므로,
그 경계에서 잘라 따로 처리해도 **결과가 정확히 같다.** 핸드아웃 힌트가 짚는 지점이 바로 이것이다.

### (2) pair count 증분 업데이트 ← **핵심**

관찰: merge `(A, B) → AB`를 적용해도 **그 pair를 포함한 단어들만** 카운트가 변한다.
나머지 99%는 그대로다. 그런데 naive는 매번 전부 다시 센다.

필요한 자료구조 두 개:

```python
pair_counts:   dict[pair, int]              # 전역 pair 빈도
pair_to_words: dict[pair, set[word_id]]     # 이 pair를 포함한 단어들 (역인덱스)
```

merge 시 하는 일:

1. `pair_to_words[best]`로 **영향받는 단어만** 가져온다
2. 그 단어들이 기존에 기여하던 pair 카운트를 **뺀다**
3. 단어를 merge된 새 심볼 열로 교체한다
4. 새 심볼 열이 만드는 pair 기여분을 **더한다**. 역인덱스도 함께 갱신
5. 소멸한 pair는 `pair_counts`에서 제거

전체 코퍼스 재스캔이 사라진다. 이것만으로 대부분 목표 시간에 들어간다.

### (3) 최빈 pair 찾기

`max()`로 매번 전체 dict를 훑는 것도 pair 종류 수에 비례한다. heap을 쓰되
stale 엔트리는 lazy deletion으로 처리하는 방식이 무난하다.
동점 규칙 때문에 정렬 키 설계에 주의해야 한다 (count는 내림차순, pair는 사전순 내림차순).

> 다만 **(1)+(2)를 먼저 하고 프로파일링해서** 여기가 실제 병목인지 확인할 것.
> 아닐 가능성이 높고, 그러면 복잡도만 추가하는 셈이다.

### 작업 순서 (권장)

1. naive로 정답 확보 → 작은 fixture로 `pytest` 통과시키기
2. (1) 병렬 pre-tokenization 추가
3. (2) 증분 업데이트로 merge 루프 교체 → **naive와 출력이 완전히 동일한지 대조**
4. `cProfile`로 남은 병목 확인 → 필요할 때만 (3)

In [ ]:
# 프로파일링 템플릿 — 실제 구현에 그대로 붙여 쓸 것
import cProfile, pstats, io

def profile(fn, *args, top=15, **kwargs):
    pr = cProfile.Profile()
    pr.enable()
    result = fn(*args, **kwargs)
    pr.disable()
    s = io.StringIO()
    pstats.Stats(pr, stream=s).sort_stats("cumulative").print_stats(top)
    print(s.getvalue())
    return result

_ = profile(train_bpe_naive, base, 50)

### 최적화 구현 검증용 셀 (구현 후 여기에 채울 것)

```python
from cs336_basics.tokenizer.train_bpe import train_bpe

# 같은 입력에 대해 naive와 실제 구현의 merges가 완전히 동일해야 한다
vocab_fast, merges_fast = train_bpe(input_path, vocab_size=..., special_tokens=[...])
assert merges_fast == merges_naive, "최적화 과정에서 로직이 바뀌었다"
```

특히 **동점 처리**가 최적화 과정에서 조용히 깨지기 쉽다.
heap이나 역인덱스를 도입하면 tie-break 순서가 미묘하게 달라지는 경우가 흔하다.

---
# 5. 정리 & 다음 단계

## 이 노트북에서 확정한 것

- `unicode1` (a)(b)(c) 답안 → `writeup/`으로 이동
- `unicode2` (a)(b)(c) 답안 → `writeup/`으로 이동
- BPE 학습 알고리즘의 동작 원리와 두 규칙 (동점 처리 / 특수 토큰 하드 경계)
- 왜 naive가 느린지, 무엇을 고쳐야 하는지

## 다음 할 일

- [ ] `cs336_basics/tokenizer/pretokenize.py` — GPT-2 regex, 특수 토큰 split, 청크 병렬화
- [ ] `cs336_basics/tokenizer/train_bpe.py` — 증분 업데이트 merge 루프
- [ ] `tests/adapters.py`의 `run_train_bpe`에서 위 구현을 **호출만** (로직 넣지 말 것)
- [ ] `uv run pytest tests/test_train_bpe.py` 통과 (속도 포함)
- [ ] 위 검증 셀에 실제 구현을 붙여 naive와 대조
- [ ] TinyStories(vocab 10K) 학습 → vocab·merges 직렬화
- [ ] `train_bpe_tinystories` 서술형: 소요 시간/메모리, 가장 긴 토큰은 무엇이고 말이 되는가, 프로파일링 결과
- [ ] OpenWebText(vocab 32K) 학습 → `train_bpe_expts_owt` 서술형

## 이어서

→ `02_tokenizer_experiments.ipynb`
   (`Tokenizer` 클래스 구현, 압축률 bytes/token 측정, 전체 데이터셋 → uint16 `.npy` 인코딩)